# BTC 1-Year Regime-Aware + Meta-Label + Selective Forecasting

This notebook is a **new experiment**, separate from the existing reproducible prediction notebook.

It uses exactly **one year of BTC market history + one year of corrected BTC news sentiment** and tests the strategies discussed for improving *reliable* directional forecasting rather than forcing an UP/DOWN prediction every hour.

## Strategy

1. **Meaningful-move target**
   - `UP` when the future return is above a causal volatility-adjusted threshold.
   - `DOWN` when it is below the negative threshold.
   - `NEUTRAL` otherwise.
   - This avoids treating a +0.01% move as equally meaningful as a +2% move.

2. **Regime-aware specialists**
   - Causal regimes are derived only from information available at time `t`.
   - Global classifier + specialist classifiers for bull / bear / range / high-volatility regimes.

3. **Multimodal features**
   - Existing market features.
   - Corrected hourly BTC sentiment.
   - Causal sentiment dynamics such as article velocity, sentiment change, rolling sentiment, disagreement and sentiment volatility.

4. **Meta-labeling**
   - A second model learns **when the primary prediction is likely to be correct**.
   - Meta labels are built from time-ordered out-of-fold primary predictions to avoid training on in-sample correctness.

5. **Abstention / selective prediction**
   - The system may return `NO SIGNAL`.
   - The confidence threshold is chosen on the validation period only.
   - The final test period remains untouched.
   - The notebook searches for the highest-coverage validation threshold that achieves the requested target accuracy (default 70%). If 70% is not achieved, it reports that honestly.

## Evaluation

For each horizon (1h, 6h, 24h), report both:

- **all-row 3-class performance**
- **selective directional accuracy**
- **coverage** = fraction of all eligible timestamps on which the model issues UP/DOWN
- class balance, confusion matrix and calibration diagnostics

> Important: 70% is a research target, not a guaranteed result. The notebook does not tune on the final test set.


In [ ]:
from google.colab import drive
drive.mount("/content/drive")


In [ ]:
!pip install -q pandas numpy pyarrow scikit-learn matplotlib joblib


## 1. Configuration

The market file is the existing BTC training-ready hourly dataset.  
The notebook filters it to exactly one year.

The sentiment file must be the **corrected 1-year sentiment output** created by the new historical sentiment notebook. This intentionally avoids using the older truncated sentiment file.


In [ ]:
from pathlib import Path
import json, math, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.base import clone
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, f1_score, balanced_accuracy_score,
    classification_report, confusion_matrix, roc_auc_score
)
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

warnings.filterwarnings("ignore")

# -------- Date window --------
START = pd.Timestamp("2025-09-04 00:00:00", tz="UTC")
END   = pd.Timestamp("2026-09-04 23:59:59", tz="UTC")

# -------- Existing project data --------
MARKET_PATH = Path(
    "/content/drive/MyDrive/crypto historical 2 year hourly data/data/"
    "04_btc_training_ready.parquet"
)

SENTIMENT_DIR = Path(
    "/content/drive/MyDrive/btc sentiment year data/data_1year_corrected"
)
SENTIMENT_PATH = SENTIMENT_DIR / "04_btc_sentiment_hourly.parquet"
SENTIMENT_MANIFEST = SENTIMENT_DIR / "run_manifest.json"

# -------- Experiment --------
HORIZONS = [1, 6, 24]
TARGET_ACCURACY = 0.70

# Meaningful-move threshold:
# threshold_h(t) = max(MIN_ABS_RETURN, VOL_MULT * vol_24h(t) * sqrt(h))
VOL_MULT = 0.35
MIN_ABS_RETURN = 0.0010   # 0.10%

# Minimum rows required before training a regime specialist.
MIN_REGIME_ROWS = 250

# Meta-model/selective threshold grid.
META_THRESHOLDS = np.round(np.arange(0.50, 0.96, 0.02), 2)
PRIMARY_CONF_THRESHOLDS = np.round(np.arange(0.40, 0.91, 0.05), 2)

# Chronological split.
TRAIN_FRAC = 0.60
VAL_FRAC = 0.20
TEST_FRAC = 0.20

OUTPUT_DIR = Path(
    "/content/drive/MyDrive/BTC reproducible multimodal pipeline/"
    "outputs/BTC_1Y_regime_metalabel_selective"
)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Market:", MARKET_PATH)
print("Sentiment:", SENTIMENT_PATH)
print("Output:", OUTPUT_DIR)

assert MARKET_PATH.exists(), f"Missing market file: {MARKET_PATH}"
assert SENTIMENT_PATH.exists(), (
    f"Missing corrected 1-year sentiment file: {SENTIMENT_PATH}\n"
    "Run Section A of run_btc_sentiment_1y_2y_corrected_colab.ipynb first."
)
assert SENTIMENT_MANIFEST.exists(), f"Missing corrected sentiment manifest: {SENTIMENT_MANIFEST}"


## 2. Load and validate the one-year market + sentiment window

Sentiment output contains rows only for hours with accepted news.  
After corrected source coverage has been verified, absent sentiment hours are represented as **no accepted news**, not as neutral news:

- `has_news = 0`
- counts = `0`
- sentiment scores/shares = `0`

This notebook does not interpolate sentiment.


In [ ]:
market = pd.read_parquet(MARKET_PATH).copy()
sent = pd.read_parquet(SENTIMENT_PATH).copy()
manifest = json.load(open(SENTIMENT_MANIFEST))

market["timestamp"] = pd.to_datetime(market["timestamp"], utc=True)
sent["timestamp"] = pd.to_datetime(sent["timestamp"], utc=True)

market = market[market["timestamp"].between(START, END)].sort_values("timestamp").reset_index(drop=True)
sent = sent[sent["timestamp"].between(START, END)].sort_values("timestamp").reset_index(drop=True)

print("Sentiment manifest requested window:")
print(" ", manifest.get("start_utc"), "->", manifest.get("end_utc"))
print()
print("Market rows:", len(market), market["timestamp"].min(), "->", market["timestamp"].max())
print("News-hour rows:", len(sent), sent["timestamp"].min(), "->", sent["timestamp"].max())

assert market["timestamp"].duplicated().sum() == 0
assert sent["timestamp"].duplicated().sum() == 0
assert market["timestamp"].is_monotonic_increasing

# Market continuity inside the available one-year training-ready slice.
full_idx = pd.date_range(
    market["timestamp"].min(),
    market["timestamp"].max(),
    freq="h",
    tz="UTC",
)
missing_market_hours = full_idx.difference(pd.DatetimeIndex(market["timestamp"]))
print("Missing market hours:", len(missing_market_hours))
assert len(missing_market_hours) == 0, "Market timeline is not continuous."


In [ ]:
# Columns produced by the sentiment aggregation pipeline.
sentiment_base = [
    "article_count", "mean_confidence", "mean_quality",
    "positive_count", "negative_count", "neutral_count",
    "unique_sources", "sentiment_score",
    "positive_share", "negative_share", "neutral_share",
]

missing_cols = [c for c in sentiment_base if c not in sent.columns]
assert not missing_cols, f"Missing sentiment columns: {missing_cols}"

df = market.merge(
    sent[["timestamp"] + sentiment_base],
    on="timestamp",
    how="left",
    validate="one_to_one",
)

# Correct no-news semantics inside the verified corrected collection window.
df["has_news"] = df["article_count"].notna().astype(np.int8)

for c in sentiment_base:
    df[c] = df[c].fillna(0.0)

print("Merged rows:", len(df))
print("Hours with accepted news:", int(df["has_news"].sum()))
print("No-accepted-news hours:", int((df["has_news"] == 0).sum()))
display(df[["timestamp","close","has_news","article_count","sentiment_score"]].head(12))


## 3. Causal sentiment/event features

These use only present and past sentiment values. No future information is used.

The current historical sources do not provide a reliable event taxonomy such as ETF / regulation / hack for every article, so this notebook does **not invent one**. Instead it adds event-intensity features that are supported by the available data.


In [ ]:
def add_sentiment_dynamics(x: pd.DataFrame) -> pd.DataFrame:
    x = x.copy()

    # Lags
    for h in (1, 6, 24):
        x[f"sentiment_score_lag_{h}h"] = x["sentiment_score"].shift(h)

    # Causal rolling means
    for h in (3, 6, 12, 24):
        x[f"sentiment_mean_{h}h"] = x["sentiment_score"].rolling(h, min_periods=1).mean()

    # Sentiment changes / surprise
    for h in (3, 6):
        x[f"sentiment_change_{h}h"] = (
            x["sentiment_score"] - x["sentiment_score"].shift(h)
        )

    # News intensity / velocity
    x["article_count_6h"] = x["article_count"].rolling(6, min_periods=1).sum()
    x["article_count_24h"] = x["article_count"].rolling(24, min_periods=1).sum()
    x["article_velocity_6h"] = x["article_count_6h"] - x["article_count_6h"].shift(6)

    # Rolling directional composition
    total6 = x["article_count"].rolling(6, min_periods=1).sum().replace(0, np.nan)
    x["positive_share_6h"] = (
        x["positive_count"].rolling(6, min_periods=1).sum() / total6
    ).fillna(0)
    x["negative_share_6h"] = (
        x["negative_count"].rolling(6, min_periods=1).sum() / total6
    ).fillna(0)

    # Sentiment disagreement. High when positive/negative/neutral shares are mixed.
    shares = x[["positive_share","negative_share","neutral_share"]].clip(0,1)
    x["sentiment_disagreement"] = 1.0 - shares.max(axis=1)

    # Causal sentiment volatility.
    x["sentiment_volatility_24h"] = (
        x["sentiment_score"].rolling(24, min_periods=2).std(ddof=0).fillna(0)
    )

    # Source breadth and confidence dynamics.
    x["source_breadth_6h"] = x["unique_sources"].rolling(6, min_periods=1).sum()
    x["confidence_mean_6h"] = x["mean_confidence"].rolling(6, min_periods=1).mean()

    return x

df = add_sentiment_dynamics(df)


## 4. Causal market regimes

Regimes are explanations/conditioning variables, not future labels.

- `high_vol`: current 24h volatility is materially above its causal historical reference.
- `bull`: positive 24h momentum and short moving average above long moving average.
- `bear`: negative 24h momentum and short moving average below long moving average.
- `range`: everything else.

The rolling volatility reference uses only information available up to time `t`.


In [ ]:
def add_causal_regime(x: pd.DataFrame) -> pd.DataFrame:
    x = x.copy()

    required = ["volatility_24h", "momentum_24h", "sma_24", "sma_168"]
    missing = [c for c in required if c not in x.columns]
    assert not missing, f"Market data missing regime columns: {missing}"

    # Expanding historical reference shifted by one hour.
    vol_ref = (
        x["volatility_24h"]
        .expanding(min_periods=168)
        .quantile(0.75)
        .shift(1)
    )

    regime = np.full(len(x), "range", dtype=object)

    high_vol = x["volatility_24h"] > vol_ref
    bull = (
        (x["momentum_24h"] > 0)
        & (x["sma_24"] > x["sma_168"])
        & (~high_vol.fillna(False))
    )
    bear = (
        (x["momentum_24h"] < 0)
        & (x["sma_24"] < x["sma_168"])
        & (~high_vol.fillna(False))
    )

    regime[high_vol.fillna(False).values] = "high_vol"
    regime[bull.fillna(False).values] = "bull"
    regime[bear.fillna(False).values] = "bear"

    x["regime"] = regime
    return x

df = add_causal_regime(df)
print(df["regime"].value_counts(dropna=False))


## 5. Meaningful-move labels

For horizon `h`, the threshold is calculated at time `t`:

`threshold(t,h) = max(0.10%, 0.35 × current_24h_volatility × sqrt(h))`

Then:

- `DOWN = 0`
- `NEUTRAL = 1`
- `UP = 2`

The threshold itself uses only information at `t`; the future return is used only to construct the supervised target.


In [ ]:
LABEL_NAMES = {0: "DOWN", 1: "NEUTRAL", 2: "UP"}

def add_target(x: pd.DataFrame, horizon: int) -> pd.DataFrame:
    z = x.copy()

    z["future_return"] = np.log(
        z["close"].shift(-horizon) / z["close"]
    )

    vol_now = z["volatility_24h"].clip(lower=0)
    z["move_threshold"] = np.maximum(
        MIN_ABS_RETURN,
        VOL_MULT * vol_now * np.sqrt(horizon),
    )

    z["target"] = np.select(
        [
            z["future_return"] < -z["move_threshold"],
            z["future_return"] >  z["move_threshold"],
        ],
        [0, 2],
        default=1,
    ).astype(float)

    z.loc[z["future_return"].isna(), "target"] = np.nan
    return z


## 6. Feature sets and strict chronological split

The final test block is not used for:

- fitting models
- selecting feature thresholds
- selecting confidence thresholds
- choosing the 70% selective-accuracy operating point


In [ ]:
MARKET_FEATURES = [
    "open","high","low","close","volume","quote_volume","num_trades",
    "log_return","log_return_lag_1h","log_return_lag_6h","log_return_lag_24h",
    "sma_24","sma_168","macd","rsi_14","momentum_6h","momentum_24h",
    "volatility_24h","volatility_168h","atr_pct_14","bb_width_20",
    "volume_zscore_24","taker_buy_ratio","buy_sell_imbalance",
    "range_pct","body_pct","close_location",
]

SENTIMENT_FEATURES = [
    "article_count","sentiment_score","positive_share","negative_share","neutral_share",
    "mean_confidence","has_news",
    "sentiment_score_lag_1h","sentiment_score_lag_6h","sentiment_score_lag_24h",
    "sentiment_mean_3h","sentiment_mean_6h","sentiment_mean_12h","sentiment_mean_24h",
    "sentiment_change_3h","sentiment_change_6h",
    "article_count_6h","article_count_24h",
    "positive_share_6h","negative_share_6h","sentiment_volatility_24h",
    "article_velocity_6h","sentiment_disagreement","source_breadth_6h","confidence_mean_6h",
]

FEATURES = [c for c in MARKET_FEATURES + SENTIMENT_FEATURES if c in df.columns]
print("Feature count:", len(FEATURES))

def chronological_split(z: pd.DataFrame, horizon: int):
    z = z.dropna(subset=FEATURES + ["target"]).reset_index(drop=True)

    n = len(z)
    i1 = int(n * TRAIN_FRAC)
    i2 = int(n * (TRAIN_FRAC + VAL_FRAC))

    # Purge around boundaries to reduce overlap with future targets.
    purge = max(48, horizon)

    train = z.iloc[:max(0, i1 - purge)].copy()
    val = z.iloc[min(n, i1 + purge):max(i1 + purge, i2 - purge)].copy()
    test = z.iloc[min(n, i2 + purge):].copy()

    assert train["timestamp"].max() < val["timestamp"].min()
    assert val["timestamp"].max() < test["timestamp"].min()

    return train, val, test


## 7. Primary regime-aware classifier

A global HistGradientBoosting model is always fitted.  
A specialist model is fitted for each regime when enough training rows are available.

At inference time:

- use the specialist for the current causal regime when available;
- otherwise fall back to the global model.

The classifier predicts the 3-class target: DOWN / NEUTRAL / UP.


In [ ]:
def new_primary_model():
    return HistGradientBoostingClassifier(
        learning_rate=0.05,
        max_iter=250,
        max_leaf_nodes=31,
        min_samples_leaf=35,
        l2_regularization=1.0,
        random_state=42,
    )

def fit_regime_models(train: pd.DataFrame):
    global_model = new_primary_model()
    global_model.fit(train[FEATURES], train["target"].astype(int))

    specialists = {}
    for regime, g in train.groupby("regime"):
        if len(g) >= MIN_REGIME_ROWS and g["target"].nunique() >= 2:
            m = new_primary_model()
            m.fit(g[FEATURES], g["target"].astype(int))
            specialists[regime] = m
            print(f"specialist {regime}: {len(g):,} rows")
        else:
            print(f"specialist {regime}: skipped ({len(g):,} rows)")

    return global_model, specialists

def _align_proba(model, X):
    raw = model.predict_proba(X)
    out = np.zeros((len(X), 3), dtype=float)
    for j, cls in enumerate(model.classes_.astype(int)):
        out[:, cls] = raw[:, j]
    return out

def predict_regime_models(global_model, specialists, z: pd.DataFrame):
    probs = np.zeros((len(z), 3), dtype=float)

    for regime, idx in z.groupby("regime").groups.items():
        idx = list(idx)
        model = specialists.get(regime, global_model)
        probs[idx] = _align_proba(model, z.loc[idx, FEATURES])

    pred = probs.argmax(axis=1)
    return pred, probs


## 8. Time-ordered OOF predictions for the meta-label model

The meta model must not learn that an in-sample primary prediction is “reliable.”

So the primary model is repeatedly trained on an earlier block and predicts a later block.  
Those out-of-fold predictions become the meta-model training data.


In [ ]:
def expanding_oof_primary(train: pd.DataFrame, n_splits: int = 5):
    n = len(train)
    min_train = max(1200, int(n * 0.35))
    remaining = n - min_train
    fold_size = max(200, remaining // n_splits)

    rows = []

    for fold in range(n_splits):
        fit_end = min_train + fold * fold_size
        pred_start = fit_end
        pred_end = n if fold == n_splits - 1 else min(n, pred_start + fold_size)

        if pred_start >= n or pred_end <= pred_start:
            continue

        fit_df = train.iloc[:fit_end].copy()
        pred_df = train.iloc[pred_start:pred_end].copy().reset_index(drop=True)

        gm, sm = fit_regime_models(fit_df)
        pred, prob = predict_regime_models(gm, sm, pred_df)

        part = pred_df[[
            "timestamp","target","regime","volatility_24h",
            "sentiment_score","article_count","has_news",
            "sentiment_disagreement","article_velocity_6h"
        ]].copy()

        part["primary_pred"] = pred
        part["p_down"] = prob[:,0]
        part["p_neutral"] = prob[:,1]
        part["p_up"] = prob[:,2]
        part["primary_conf"] = prob.max(axis=1)

        sorted_p = np.sort(prob, axis=1)
        part["prob_margin"] = sorted_p[:,-1] - sorted_p[:,-2]
        part["prob_entropy"] = -np.sum(
            np.clip(prob,1e-12,1) * np.log(np.clip(prob,1e-12,1)),
            axis=1,
        )

        part["meta_target"] = (
            part["primary_pred"].astype(int) == part["target"].astype(int)
        ).astype(int)

        rows.append(part)

    assert rows, "Could not build OOF predictions."
    return pd.concat(rows, ignore_index=True)


## 9. Meta-label model

The meta-model estimates:

> `P(primary prediction is correct | confidence, regime, volatility, sentiment context, disagreement, news intensity)`

This probability is **not** the same as `P(BTC goes up)`.


In [ ]:
REGIME_DUMMIES = ["bull","bear","range","high_vol"]

def make_meta_X(z: pd.DataFrame):
    x = z.copy()

    for r in REGIME_DUMMIES:
        x[f"regime_{r}"] = (x["regime"] == r).astype(int)

    cols = [
        "primary_conf","prob_margin","prob_entropy",
        "p_down","p_neutral","p_up",
        "volatility_24h","sentiment_score","article_count","has_news",
        "sentiment_disagreement","article_velocity_6h",
    ] + [f"regime_{r}" for r in REGIME_DUMMIES]

    return x[cols].replace([np.inf,-np.inf], np.nan).fillna(0.0)

def fit_meta_model(oof: pd.DataFrame):
    meta = Pipeline([
        ("scale", StandardScaler()),
        ("model", LogisticRegression(
            max_iter=2000,
            class_weight="balanced",
            random_state=42,
        )),
    ])
    meta.fit(make_meta_X(oof), oof["meta_target"].astype(int))
    return meta

def build_primary_frame(z: pd.DataFrame, pred, prob):
    out = z[[
        "timestamp","target","regime","volatility_24h",
        "sentiment_score","article_count","has_news",
        "sentiment_disagreement","article_velocity_6h",
    ]].copy().reset_index(drop=True)

    out["primary_pred"] = pred
    out["p_down"] = prob[:,0]
    out["p_neutral"] = prob[:,1]
    out["p_up"] = prob[:,2]
    out["primary_conf"] = prob.max(axis=1)

    sorted_p = np.sort(prob, axis=1)
    out["prob_margin"] = sorted_p[:,-1] - sorted_p[:,-2]
    out["prob_entropy"] = -np.sum(
        np.clip(prob,1e-12,1) * np.log(np.clip(prob,1e-12,1)),
        axis=1,
    )
    return out


## 10. Selective operating-point selection on validation only

A signal is issued only when:

- primary prediction is `UP` or `DOWN`;
- primary class confidence exceeds a threshold;
- meta-model reliability exceeds a threshold.

Among validation settings reaching the target accuracy, choose the one with **highest coverage**.

If none reach 70%, choose the best validation trade-off and explicitly mark `target_reached=False`.


In [ ]:
def selective_metrics(frame, meta_prob, meta_thr, primary_thr):
    y = frame["target"].astype(int).to_numpy()
    pred = frame["primary_pred"].astype(int).to_numpy()

    signal = (
        (pred != 1) &
        (frame["primary_conf"].to_numpy() >= primary_thr) &
        (meta_prob >= meta_thr)
    )

    n_signal = int(signal.sum())
    coverage = float(signal.mean())

    if n_signal == 0:
        return {
            "accuracy": np.nan,
            "coverage": 0.0,
            "signals": 0,
            "directional_f1": np.nan,
        }

    acc = accuracy_score(y[signal], pred[signal])
    f1 = f1_score(
        y[signal], pred[signal],
        labels=[0,2], average="macro", zero_division=0
    )

    return {
        "accuracy": float(acc),
        "coverage": coverage,
        "signals": n_signal,
        "directional_f1": float(f1),
    }

def select_operating_point(val_frame, val_meta_prob, target_accuracy=0.70):
    rows = []
    for mt in META_THRESHOLDS:
        for pt in PRIMARY_CONF_THRESHOLDS:
            m = selective_metrics(val_frame, val_meta_prob, mt, pt)
            rows.append({
                "meta_threshold": float(mt),
                "primary_threshold": float(pt),
                **m,
            })

    table = pd.DataFrame(rows)
    feasible = table[
        (table["signals"] >= 30) &
        (table["accuracy"] >= target_accuracy)
    ].copy()

    if len(feasible):
        chosen = feasible.sort_values(
            ["coverage","accuracy"],
            ascending=[False,False],
        ).iloc[0].to_dict()
        chosen["target_reached"] = True
    else:
        candidates = table[table["signals"] >= 30].copy()
        if not len(candidates):
            candidates = table[table["signals"] > 0].copy()

        # Accuracy first, then coverage. Test data is never used here.
        chosen = candidates.sort_values(
            ["accuracy","coverage"],
            ascending=[False,False],
        ).iloc[0].to_dict()
        chosen["target_reached"] = False

    return chosen, table


## 11. Run one horizon

This function performs the complete experiment for one horizon:

1. construct meaningful-move labels;
2. split chronologically;
3. create OOF primary predictions;
4. train meta-label model;
5. train final primary model on training block;
6. tune abstention thresholds on validation block;
7. evaluate the frozen operating point once on untouched test data.


In [ ]:
def run_horizon(horizon: int):
    print("\n" + "="*80)
    print(f"HORIZON: {horizon}h")
    print("="*80)

    z = add_target(df, horizon)
    train, val, test = chronological_split(z, horizon)

    print("train:", len(train), train["timestamp"].min(), "->", train["timestamp"].max())
    print("val:  ", len(val), val["timestamp"].min(), "->", val["timestamp"].max())
    print("test: ", len(test), test["timestamp"].min(), "->", test["timestamp"].max())

    print("\nTarget balance:")
    for name, part in [("train",train),("val",val),("test",test)]:
        print(name, part["target"].astype(int).map(LABEL_NAMES).value_counts(normalize=True).round(3).to_dict())

    # OOF meta-training set.
    oof = expanding_oof_primary(train, n_splits=5)
    meta = fit_meta_model(oof)

    # Final primary models trained only on train.
    global_model, specialists = fit_regime_models(train)

    val_pred, val_prob = predict_regime_models(global_model, specialists, val.reset_index(drop=True))
    test_pred, test_prob = predict_regime_models(global_model, specialists, test.reset_index(drop=True))

    val_frame = build_primary_frame(val.reset_index(drop=True), val_pred, val_prob)
    test_frame = build_primary_frame(test.reset_index(drop=True), test_pred, test_prob)

    val_meta_prob = meta.predict_proba(make_meta_X(val_frame))[:,1]
    test_meta_prob = meta.predict_proba(make_meta_X(test_frame))[:,1]

    # All-row primary metrics.
    val_all_acc = accuracy_score(val_frame["target"].astype(int), val_frame["primary_pred"].astype(int))
    test_all_acc = accuracy_score(test_frame["target"].astype(int), test_frame["primary_pred"].astype(int))
    test_bal_acc = balanced_accuracy_score(test_frame["target"].astype(int), test_frame["primary_pred"].astype(int))

    chosen, grid = select_operating_point(val_frame, val_meta_prob, TARGET_ACCURACY)

    test_sel = selective_metrics(
        test_frame,
        test_meta_prob,
        chosen["meta_threshold"],
        chosen["primary_threshold"],
    )

    summary = {
        "horizon_h": horizon,
        "val_all_3class_accuracy": float(val_all_acc),
        "test_all_3class_accuracy": float(test_all_acc),
        "test_all_balanced_accuracy": float(test_bal_acc),
        "selected_meta_threshold": float(chosen["meta_threshold"]),
        "selected_primary_threshold": float(chosen["primary_threshold"]),
        "validation_selective_accuracy": float(chosen["accuracy"]) if pd.notna(chosen["accuracy"]) else None,
        "validation_selective_coverage": float(chosen["coverage"]),
        "validation_target_70_reached": bool(chosen["target_reached"]),
        "test_selective_accuracy": float(test_sel["accuracy"]) if pd.notna(test_sel["accuracy"]) else None,
        "test_selective_coverage": float(test_sel["coverage"]),
        "test_selective_signals": int(test_sel["signals"]),
        "test_selective_directional_f1": float(test_sel["directional_f1"]) if pd.notna(test_sel["directional_f1"]) else None,
    }

    print("\nChosen on validation:")
    print(json.dumps(chosen, indent=2, default=float))

    print("\nUntouched test:")
    print(json.dumps(summary, indent=2))

    print("\nAll-row 3-class test classification report:")
    print(classification_report(
        test_frame["target"].astype(int),
        test_frame["primary_pred"].astype(int),
        labels=[0,1,2],
        target_names=["DOWN","NEUTRAL","UP"],
        zero_division=0,
    ))

    # Save details.
    hdir = OUTPUT_DIR / f"{horizon}h"
    hdir.mkdir(parents=True, exist_ok=True)
    pd.DataFrame([summary]).to_csv(hdir / "summary.csv", index=False)
    grid.to_csv(hdir / "validation_threshold_grid.csv", index=False)

    test_out = test_frame.copy()
    test_out["meta_reliability"] = test_meta_prob
    test_out["issue_signal"] = (
        (test_out["primary_pred"] != 1) &
        (test_out["primary_conf"] >= chosen["primary_threshold"]) &
        (test_out["meta_reliability"] >= chosen["meta_threshold"])
    )
    test_out["decision"] = np.where(
        test_out["issue_signal"],
        test_out["primary_pred"].map(LABEL_NAMES),
        "NO SIGNAL",
    )
    test_out.to_parquet(hdir / "test_predictions.parquet", index=False)
    test_out.to_csv(hdir / "test_predictions.csv", index=False)

    return summary, grid, test_out


## 12. Run 1h, 6h and 24h

Start with all three horizons because your previous experiments suggest predictive usefulness can be horizon-dependent.

The notebook does **not** assume the 1h horizon must be best.


In [ ]:
all_results = []
all_test_predictions = {}

for h in HORIZONS:
    summary, grid, test_predictions = run_horizon(h)
    all_results.append(summary)
    all_test_predictions[h] = test_predictions

results = pd.DataFrame(all_results)
results.to_csv(OUTPUT_DIR / "all_horizon_summary.csv", index=False)
display(results)


## 13. Accuracy vs coverage

This is the key interpretation for selective prediction.

A legitimate result might look like:

- 100% coverage → 55% accuracy
- 40% coverage → 64% accuracy
- 20% coverage → 71% accuracy

The exact numbers must come from the untouched test set.  
If the test accuracy does not reach 70%, do not change the threshold using test labels.


In [ ]:
for h in HORIZONS:
    pred = all_test_predictions[h]
    row = results.loc[results["horizon_h"] == h].iloc[0]

    print(
        f"{h}h | all-row 3-class acc={row['test_all_3class_accuracy']:.3f} | "
        f"selective acc={row['test_selective_accuracy']} | "
        f"coverage={row['test_selective_coverage']:.3f} | "
        f"signals={int(row['test_selective_signals'])}"
    )


## 14. Optional: inspect signal quality by regime

This helps answer whether selective forecasting works only in particular market states.

Do not use this post-hoc test analysis to retune the current test set. Treat it as diagnostic evidence for a later experiment.


In [ ]:
for h, pred in all_test_predictions.items():
    sig = pred[pred["issue_signal"]].copy()
    if not len(sig):
        continue

    sig["correct"] = (sig["primary_pred"].astype(int) == sig["target"].astype(int)).astype(int)

    by_regime = sig.groupby("regime").agg(
        signals=("correct","size"),
        accuracy=("correct","mean"),
        mean_primary_conf=("primary_conf","mean"),
        mean_meta_reliability=("meta_reliability","mean"),
    ).sort_values("signals", ascending=False)

    print(f"\n{h}h selective signals by regime")
    display(by_regime)


# Interpretation rules

### A strong result
A strong result is **not** simply `test accuracy > 70%`.

It should jointly show:

- selective accuracy materially above the baseline;
- enough coverage to be useful;
- performance that survives the untouched chronological test;
- reasonable results across regimes rather than one tiny lucky cluster;
- no timestamp leakage;
- stable behavior when the experiment is repeated on a future period.

### If 70% is reached only at 2–5% coverage
Report it, but do not claim that the model predicts BTC with 70% overall accuracy.

Correct wording:

> “At the validation-selected operating point, the selective forecaster achieved X% directional accuracy while issuing signals on Y% of test timestamps.”

### If 70% is not reached
That is also a valid research result. Do not tune on the test set.

Possible next experiments:

1. add funding rate / open interest / liquidations;
2. add event-type extraction from article-level news;
3. compare the regime-aware tree ensemble with a multimodal sequence Transformer;
4. repeat on a later forward-test period;
5. calibrate the meta-reliability model.
